In [1]:
# Librerías
from pathlib import Path
import pandas as pd
import numpy as np

In [4]:
# Raíz del proyecto
PROJECT_ROOT = Path().resolve().parents[1]

# Rutas de entrada
path_dep = PROJECT_ROOT / "data" / "middle" / "produccion_caldas_2012-2019.csv"
path_mun = PROJECT_ROOT / "data" / "middle" / "produccion_caldas_municipal_2019-2024.csv"

print("Departamental:", path_dep)
print("Municipal:", path_mun)

Departamental: /home/ddayann/proyectos/Coffe/proyecto_aplicado_en_analitica_de_datos/data/middle/produccion_caldas_2012-2019.csv
Municipal: /home/ddayann/proyectos/Coffe/proyecto_aplicado_en_analitica_de_datos/data/middle/produccion_caldas_municipal_2019-2024.csv


In [5]:
# Cargar bases
dep = pd.read_csv(path_dep)
mun = pd.read_csv(path_mun)

print("Base departamental:", dep.shape)
print("Base municipal:", mun.shape)

display(dep.head())
display(mun.head())

Base departamental: (8, 7)
Base municipal: (150, 7)


,Codigo,Departamento,Año,Area_plantada,Area_productiva,Produccion,Rendimiento
0,17,Caldas,2012,63153.048130,39557.135528,62517.871130,1.580445
1,17,Caldas,2013,78518.811344,55007.637172,99537.676621,1.809525
2,17,Caldas,2014,80146.239242,57755.254949,103893.906467,1.798865
3,17,Caldas,2015,86804.311378,62553.217490,105878.111170,1.692609
4,17,Caldas,2016,66399.543425,51244.905866,102724.751149,2.004585


,Código Dane municipio,Municipio,Año,Área sembrada (ha),Área cosechada (ha),Producción (t),Rendimiento (t/ha)
0,17001,Manizales,2019,5075.00,3444.00,4597.00,1.33
1,17001,Manizales,2020,4920.00,3362.00,4149.00,1.23
2,17001,Manizales,2021,4781.70,3268.13,4934.88,1.51
3,17001,Manizales,2022,4765.43,3386.54,4076.85,1.20
4,17001,Manizales,2023,4778.52,3466.14,6275.13,1.81


In [22]:
# Calibración de índices a partir de año superopuesto
dep_caldas = dep[dep["Departamento"].astype(str).str.strip().eq("Caldas")].copy()

# Año de calibración
dep_2019 = dep_caldas[dep_caldas["Año"] == 2019].copy()
mun_2019 = mun[mun["Año"] == 2019].copy()

display(dep_2019)
display(mun_2019.head())

,Codigo,Departamento,Año,Area_plantada,Area_productiva,Produccion,Rendimiento
7,17,Caldas,2019,64324.17,49335.9,65637.93,1.330429


,Código Dane municipio,Municipio,Año,Área sembrada (ha),Área cosechada (ha),Producción (t),Rendimiento (t/ha)
0,17001,Manizales,2019,5075.0,3444.0,4597.0,1.33
6,17013,Aguadas,2019,4094.0,3079.0,4068.0,1.32
12,17042,Anserma,2019,5571.0,4099.0,5427.0,1.32
18,17050,Aranzazu,2019,1758.0,1379.0,1771.0,1.28
24,17088,Belalcázar,2019,2890.0,2085.0,2752.0,1.32


La información proveniente del ENA (Encuesta Nacional Agropecuaria) evidencia un valor de sobreajuste del 5.63% con respecto al valor EVA (Evaluación Agropecuaria Municipal) que representa un dato más preciso (capturado a partir de reportes administrativos). 
Este dato será utilizado para la calibración de las producciones de los años previos (ENA) y escalarlos a nivel municipal. 

In [28]:
# Totales municipales observados en 2019
total_area_sem_2019 = mun_2019["Área sembrada (ha)"].sum()
total_area_cos_2019 = mun_2019["Área cosechada (ha)"].sum()
total_prod_2019 = mun_2019["Producción (t)"].sum()

print(f"Total área sembrada (EVA 2019): {total_area_sem_2019}")
print(f"Total área cosechada (EVA 2019): {total_area_cos_2019}")
print(f"Total producción (EVA 2019): {total_prod_2019}")

Total área sembrada (EVA 2019): 63050.0
Total área cosechada (EVA 2019): 48468.0
Total producción (EVA 2019): 62138.4


In [37]:
# Participaciones municipales 2019
mun_2019 = mun_2019.copy()

mun_2019["part_area_sem_2019"] = mun_2019["Área sembrada (ha)"] / total_area_sem_2019
mun_2019["part_area_cos_2019"] = mun_2019["Área cosechada (ha)"] / total_area_cos_2019
mun_2019["part_prod_2019"] = mun_2019["Producción (t)"] / total_prod_2019

# Verificación
print("Suma participación área sembrada:", mun_2019["part_area_sem_2019"].sum())
print("Suma participación área cosechada:", mun_2019["part_area_cos_2019"].sum())
print("Suma participación producción:", mun_2019["part_prod_2019"].sum())

display(
    mun_2019[
        [
            "Código Dane municipio",
            "Municipio",
            "Área sembrada (ha)",
            "Área cosechada (ha)",
            "Producción (t)",
            "part_area_sem_2019",
            "part_area_cos_2019",
            "part_prod_2019"
        ]
    ].head()
)

Suma participación área sembrada: 1.0
Suma participación área cosechada: 0.9999999999999999
Suma participación producción: 1.0


,Código Dane municipio,Municipio,Área sembrada (ha),Área cosechada (ha),Producción (t),part_area_sem_2019,part_area_cos_2019,part_prod_2019
0,17001,Manizales,5075.0,3444.0,4597.0,0.080492,0.071057,0.073980
6,17013,Aguadas,4094.0,3079.0,4068.0,0.064933,0.063526,0.065467
12,17042,Anserma,5571.0,4099.0,5427.0,0.088358,0.084571,0.087337
18,17050,Aranzazu,1758.0,1379.0,1771.0,0.027883,0.028452,0.028501
24,17088,Belalcázar,2890.0,2085.0,2752.0,0.045837,0.043018,0.044288


In [38]:
# Años a estimar
anios_estimacion = list(range(2012, 2019))

# Base departamental histórica
dep_hist = dep_caldas[dep_caldas["Año"].isin(anios_estimacion)].copy()

# Seleccionar columnas relevantes
dep_hist = dep_hist[
    ["Año", "Area_plantada", "Area_productiva", "Produccion", "Rendimiento"]
].copy()

display(dep_hist)

,Año,Area_plantada,Area_productiva,Produccion,Rendimiento
0,2012,63153.048130,39557.135528,62517.871130,1.580445
1,2013,78518.811344,55007.637172,99537.676621,1.809525
2,2014,80146.239242,57755.254949,103893.906467,1.798865
3,2015,86804.311378,62553.217490,105878.111170,1.692609
4,2016,66399.543425,51244.905866,102724.751149,2.004585
5,2017,59267.670000,42685.940000,96968.580000,2.270000
6,2018,63942.470000,48614.980000,63660.440000,1.309482


In [39]:
# Base de participaciones
participaciones = mun_2019[
    [
        "Código Dane municipio",
        "Municipio",
        "part_area_sem_2019",
        "part_area_cos_2019",
        "part_prod_2019"
    ]
].copy()

estimaciones = []

for anio in anios_estimacion:
    fila_dep = dep_hist[dep_hist["Año"] == anio].iloc[0]
    
    area_plantada_dep = fila_dep["Area_plantada"]
    area_productiva_dep = fila_dep["Area_productiva"]
    produccion_dep = fila_dep["Produccion"]
    
    temp = participaciones.copy()
    temp["Año"] = anio
    
    # Estimaciones municipales
    temp["Área sembrada (ha)"] = temp["part_area_sem_2019"] * area_plantada_dep
    temp["Área cosechada (ha)"] = temp["part_area_cos_2019"] * area_productiva_dep
    temp["Producción (t)"] = temp["part_prod_2019"] * produccion_dep
    
    estimaciones.append(temp)

estim_2012_2018 = pd.concat(estimaciones, ignore_index=True)

display(estim_2012_2018.head())
print(estim_2012_2018.shape)

,Código Dane municipio,Municipio,part_area_sem_2019,part_area_cos_2019,part_prod_2019,Año,Área sembrada (ha),Área cosechada (ha),Producción (t)
0,17001,Manizales,0.080492,0.071057,0.073980,2012,5083.294516,2810.818989,4625.073281
1,17013,Aguadas,0.064933,0.063526,0.065467,2012,4100.691182,2512.924410,4092.842747
2,17042,Anserma,0.088358,0.084571,0.087337,2012,5580.105173,3345.396933,5460.141983
3,17050,Aranzazu,0.027883,0.028452,0.028501,2012,1760.873253,1125.470205,1781.815267
4,17088,Belalcázar,0.045837,0.043018,0.044288,2012,2894.723380,1701.671775,2768.806106


(175, 9)


In [42]:
# Cálculo de rendimientos municipales
estim_2012_2018["Rendimiento (t/ha)"] = np.where(
    estim_2012_2018["Área cosechada (ha)"] > 0,
    estim_2012_2018["Producción (t)"] / estim_2012_2018["Área cosechada (ha)"],
    np.nan
)

# Redondeo
estim_2012_2018["Rendimiento (t/ha)"] = estim_2012_2018["Rendimiento (t/ha)"].round(2)

display(
    estim_2012_2018[
        [
            "Código Dane municipio",
            "Municipio",
            "Año",
            "Área sembrada (ha)",
            "Área cosechada (ha)",
            "Producción (t)",
            "Rendimiento (t/ha)"
        ]
    ].head()
)

,Código Dane municipio,Municipio,Año,Área sembrada (ha),Área cosechada (ha),Producción (t),Rendimiento (t/ha)
0,17001,Manizales,2012,5083.294516,2810.818989,4625.073281,1.65
1,17013,Aguadas,2012,4100.691182,2512.924410,4092.842747,1.63
2,17042,Anserma,2012,5580.105173,3345.396933,5460.141983,1.63
3,17050,Aranzazu,2012,1760.873253,1125.470205,1781.815267,1.58
4,17088,Belalcázar,2012,2894.723380,1701.671775,2768.806106,1.63


In [45]:
# Validación de totales municipales vs EVA departamental
check_mun = (
    estim_2012_2018.groupby("Año", as_index=False)
    .agg({
        "Área sembrada (ha)": "sum",
        "Área cosechada (ha)": "sum",
        "Producción (t)": "sum"
    })
)

# Renombrar para comparación
check_mun = check_mun.rename(columns={
    "Área sembrada (ha)": "Area_sembrada_mun",
    "Área cosechada (ha)": "Area_cosechada_mun",
    "Producción (t)": "Produccion_mun"
})

check_dep = dep_hist.rename(columns={
    "Area_plantada": "Area_sembrada_dep",
    "Area_productiva": "Area_cosechada_dep",
    "Produccion": "Produccion_dep",
    "Rendimiento": "Rendimiento_dep"
}).copy()

check = check_dep.merge(check_mun, on="Año", how="left")

check["diff_area_sem"] = check["Area_sembrada_dep"] - check["Area_sembrada_mun"]
check["diff_area_cos"] = check["Area_cosechada_dep"] - check["Area_cosechada_mun"]
check["diff_prod"] = check["Produccion_dep"] - check["Produccion_mun"]

display(check)

,Año,Area_sembrada_dep,Area_cosechada_dep,Produccion_dep,Rendimiento_dep,Area_sembrada_mun,Area_cosechada_mun,Produccion_mun,diff_area_sem,diff_area_cos,diff_prod
0,2012,63153.048130,39557.135528,62517.871130,1.580445,63153.048130,39557.135528,62517.871130,0.0,0.0,0.0
1,2013,78518.811344,55007.637172,99537.676621,1.809525,78518.811344,55007.637172,99537.676621,0.0,0.0,0.0
2,2014,80146.239242,57755.254949,103893.906467,1.798865,80146.239242,57755.254949,103893.906467,0.0,0.0,0.0
3,2015,86804.311378,62553.217490,105878.111170,1.692609,86804.311378,62553.217490,105878.111170,0.0,0.0,0.0
4,2016,66399.543425,51244.905866,102724.751149,2.004585,66399.543425,51244.905866,102724.751149,0.0,0.0,0.0
5,2017,59267.670000,42685.940000,96968.580000,2.270000,59267.670000,42685.940000,96968.580000,0.0,0.0,0.0
6,2018,63942.470000,48614.980000,63660.440000,1.309482,63942.470000,48614.980000,63660.440000,0.0,0.0,0.0


In [46]:
# Validación de rendimientos municipales vs EVA departamental
rend_check = (
    estim_2012_2018.groupby("Año", as_index=False)
    .agg({
        "Área cosechada (ha)": "sum",
        "Producción (t)": "sum"
    })
)

rend_check["Rendimiento_mun_implicito"] = (
    rend_check["Producción (t)"] / rend_check["Área cosechada (ha)"]
).round(6)

# Unir con rendimiento ENA
rend_check = rend_check.merge(
    dep_hist[["Año", "Rendimiento"]].rename(columns={"Rendimiento": "Rendimiento_ENA"}),
    on="Año",
    how="left"
)

rend_check["Diferencia"] = rend_check["Rendimiento_mun_implicito"] - rend_check["Rendimiento_ENA"]

display(rend_check)

,Año,Área cosechada (ha),Producción (t),Rendimiento_mun_implicito,Rendimiento_ENA,Diferencia
0,2012,39557.135528,62517.871130,1.580445,1.580445,1.498899e-07
1,2013,55007.637172,99537.676621,1.809525,1.809525,3.278273e-07
2,2014,57755.254949,103893.906467,1.798865,1.798865,3.932888e-09
3,2015,62553.217490,105878.111170,1.692609,1.692609,4.433546e-07
4,2016,51244.905866,102724.751149,2.004585,2.004585,3.605431e-07
5,2017,42685.940000,96968.580000,2.271675,2.270000,1.675000e-03
6,2018,48614.980000,63660.440000,1.309482,1.309482,0.000000e+00


In [47]:
# Resumen de rendimientos municipales estimados
resumen_rend = (
    estim_2012_2018.groupby("Año")["Rendimiento (t/ha)"]
    .agg(["min", "max", "mean", "median", "std"])
    .round(3)
    .reset_index()
)

display(resumen_rend)

,Año,min,max,mean,median,std
0,2012,0.80,1.66,1.572,1.60,0.165
1,2013,0.92,1.90,1.800,1.83,0.188
2,2014,0.91,1.89,1.789,1.82,0.188
3,2015,0.86,1.78,1.684,1.72,0.176
4,2016,1.02,2.11,1.995,2.03,0.208
5,2017,1.15,2.39,2.260,2.30,0.237
6,2018,0.66,1.38,1.303,1.33,0.137


In [48]:
# Resumen de rendimientos observados 2019-2024
resumen_obs = (
    mun.groupby("Año")["Rendimiento (t/ha)"]
    .agg(["min", "max", "mean", "median", "std"])
    .round(3)
    .reset_index()
)

display(resumen_obs)

,Año,min,max,mean,median,std
0,2019,0.65,1.35,1.274,1.30,0.133
1,2020,0.24,1.70,1.076,1.20,0.341
2,2021,0.82,1.51,1.096,1.09,0.164
3,2022,0.52,1.88,1.046,0.98,0.247
4,2023,0.76,1.85,1.112,1.07,0.329
5,2024,0.54,2.29,1.324,1.32,0.453


In [50]:
base_final_2012_2018 = estim_2012_2018[
    [
        "Código Dane municipio",
        "Municipio",
        "Año",
        "Área sembrada (ha)",
        "Área cosechada (ha)",
        "Producción (t)",
        "Rendimiento (t/ha)"
    ]
].copy()

# Redondeos finales
base_final_2012_2018["Área sembrada (ha)"] = base_final_2012_2018["Área sembrada (ha)"].round(2)
base_final_2012_2018["Área cosechada (ha)"] = base_final_2012_2018["Área cosechada (ha)"].round(2)
base_final_2012_2018["Producción (t)"] = base_final_2012_2018["Producción (t)"].round(2)
base_final_2012_2018["Rendimiento (t/ha)"] = base_final_2012_2018["Rendimiento (t/ha)"].round(2)

base_final_2012_2018 = base_final_2012_2018.sort_values(
    ["Municipio", "Año"]
).reset_index(drop=True)

display(base_final_2012_2018.head(10))

,Código Dane municipio,Municipio,Año,Área sembrada (ha),Área cosechada (ha),Producción (t),Rendimiento (t/ha)
0,17013,Aguadas,2012,4100.69,2512.92,4092.84,1.63
1,17013,Aguadas,2013,5098.43,3494.44,6516.41,1.86
2,17013,Aguadas,2014,5204.10,3668.99,6801.60,1.85
3,17013,Aguadas,2015,5636.43,3973.78,6931.50,1.74
4,17013,Aguadas,2016,4311.49,3255.41,6725.06,2.07
5,17013,Aguadas,2017,3848.40,2711.69,6348.22,2.34
6,17013,Aguadas,2018,4151.95,3088.34,4167.64,1.35
7,17042,Anserma,2012,5580.11,3345.40,5460.14,1.63
8,17042,Anserma,2013,6937.80,4652.07,8693.35,1.87
9,17042,Anserma,2014,7081.60,4884.43,9073.81,1.86


In [52]:
# Base de referencia pra consolidación
mun_2019_2024 = mun[
    [
        "Código Dane municipio",
        "Municipio",
        "Año",
        "Área sembrada (ha)",
        "Área cosechada (ha)",
        "Producción (t)",
        "Rendimiento (t/ha)"
    ]
].copy()

In [53]:
# Base estimada
base_final_2012_2018["Código Dane municipio"] = (
    base_final_2012_2018["Código Dane municipio"].astype(str).str.zfill(5)
)
base_final_2012_2018["Municipio"] = base_final_2012_2018["Municipio"].astype(str).str.strip()
base_final_2012_2018["Año"] = pd.to_numeric(base_final_2012_2018["Año"], errors="coerce").astype(int)

# Base observada
mun_2019_2024["Código Dane municipio"] = (
    mun_2019_2024["Código Dane municipio"].astype(str).str.zfill(5)
)
mun_2019_2024["Municipio"] = mun_2019_2024["Municipio"].astype(str).str.strip()
mun_2019_2024["Año"] = pd.to_numeric(mun_2019_2024["Año"], errors="coerce").astype(int)

In [59]:
produccion_caldas = pd.concat(
    [base_final_2012_2018, mun_2019_2024],
    ignore_index=True
)
display(produccion_caldas.head(5))
display(produccion_caldas.tail(5))


,Código Dane municipio,Municipio,Año,Área sembrada (ha),Área cosechada (ha),Producción (t),Rendimiento (t/ha)
0,17013,Aguadas,2012,4100.69,2512.92,4092.84,1.63
1,17013,Aguadas,2013,5098.43,3494.44,6516.41,1.86
2,17013,Aguadas,2014,5204.10,3668.99,6801.60,1.85
3,17013,Aguadas,2015,5636.43,3973.78,6931.50,1.74
4,17013,Aguadas,2016,4311.49,3255.41,6725.06,2.07


,Código Dane municipio,Municipio,Año,Área sembrada (ha),Área cosechada (ha),Producción (t),Rendimiento (t/ha)
320,17877,Viterbo,2020,441.00,353.00,424.00,1.20
321,17877,Viterbo,2021,449.69,350.97,477.32,1.36
322,17877,Viterbo,2022,360.73,265.15,497.98,1.88
323,17877,Viterbo,2023,350.96,261.10,465.32,1.78
324,17877,Viterbo,2024,333.99,246.92,544.00,2.20


In [60]:
# Exportar archivo final
output_final = PROJECT_ROOT / "data" / "processed" / "produccion_caldas_2012_2024.csv"
produccion_caldas.to_csv(output_final, index=False, encoding="utf-8-sig")
print(f"Archivo final guardado en: {output_final}")

Archivo final guardado en: /home/ddayann/proyectos/Coffe/proyecto_aplicado_en_analitica_de_datos/data/processed/produccion_caldas_2012_2024.csv
